# Rotary Positional Embeddings (RoPE): From Scratch

**RoPE** is the standard way to handle positions in modern LLMs. It combines the best of absolute and relative positional embeddings.

In this tutorial, we will:
1. Understand the math of 2D rotations.
2. Implement RoPE manually using complex numbers.
3. Implement RoPE efficiently using real numbers (like in production).
4. Use the `torchtune` implementation.

## 1. The Math of Rotation

If we have a 2D vector $(x, y)$, we can rotate it by an angle $\theta$ using a rotation matrix:

$$ \begin{pmatrix} x' \\ y' \end{pmatrix} = \begin{pmatrix} \cos \theta & -\sin \theta \\ \sin \theta & \cos \theta \end{pmatrix} \begin{pmatrix} x \\ y \end{pmatrix} $$

In RoPE, we pair up the dimensions of our embedding vectors and rotate each pair by a different angle depending on the position $m$.

## 2. Manual Implementation

Let's implement this for a single pair of numbers.

In [ ]:
import torch

def rotate_vector(x, y, angle):
    x_new = x * torch.cos(angle) - y * torch.sin(angle)
    y_new = x * torch.sin(angle) + y * torch.cos(angle)
    return x_new, y_new

# Test rotation by 90 degrees (pi/2)
x, y = torch.tensor(1.0), torch.tensor(0.0)
angle = torch.tensor(3.14159 / 2)

x_new, y_new = rotate_vector(x, y, angle)
print(f"Original: ({x:.1f}, {y:.1f})")
print(f"Rotated:  ({x_new:.1f}, {y_new:.1f})")

## 3. Efficient Implementation

In a Transformer, we have vectors of size `d_model`. We split them into pairs and rotate each pair. The angle for position $m$ and dimension pair $i$ is:

$$ \theta_{m, i} = m \cdot 10000^{-2i/d} $$

In [ ]:
def precompute_freqs_cis(dim: int, end: int, theta: float = 10000.0):
    # Calculate frequencies: 1 / (theta ^ (2i / dim))
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    
    # Create position indices: [0, 1, ..., end-1]
    t = torch.arange(end, device=freqs.device)
    
    # Outer product to get angles for each position and frequency
    freqs = torch.outer(t, freqs).float()  # [seq_len, dim/2]
    
    # Convert to complex numbers: cos(freq) + i*sin(freq)
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)  # complex64
    return freqs_cis

def apply_rotary_emb(xq: torch.Tensor, xk: torch.Tensor, freqs_cis: torch.Tensor):
    # Reshape to complex numbers: [batch, seq, head, dim/2]
    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
    
    # Reshape freqs for broadcasting: [1, seq, 1, dim/2]
    freqs_cis = freqs_cis.view(1, xq.size(1), 1, xq.size(-1) // 2)
    
    # Rotate by complex multiplication
    xq_out = torch.view_as_real(xq_ * freqs_cis).flatten(3)
    xk_out = torch.view_as_real(xk_ * freqs_cis).flatten(3)
    
    return xq_out.type_as(xq), xk_out.type_as(xk)

# Test it
dim = 64
seq_len = 10
freqs_cis = precompute_freqs_cis(dim, seq_len)

q = torch.randn(1, seq_len, 4, dim)
k = torch.randn(1, seq_len, 4, dim)

q_rot, k_rot = apply_rotary_emb(q, k, freqs_cis)
print("Rotated Q shape:", q_rot.shape)

## 4. Production Usage

In our codebase, we use `torchtune` which handles this efficiently.

In [ ]:
from torchtune.modules import RotaryPositionalEmbeddings

rope = RotaryPositionalEmbeddings(dim=dim, max_seq_len=seq_len)
q_rot_tune = rope(q)

print("Torchtune Output:", q_rot_tune.shape)